# Point & Feature Vegetation-Index Analysis

**What / why.** This notebook reproduces, in a standalone GEE-authenticated environment, three
RAVI analyses that originally lived inside the QGIS plugin dialog:

1. **Per-point vegetation-index time series** (the coordinate-capture workflow): for each captured
   point we extract an NDVI time series from the Sentinel-2 collection.
2. **Per-feature time series keyed by an ID attribute**: each polygon of an input vector layer is
   treated as its own AOI and gets its own AOI-mean NDVI series, labelled by an `id` attribute.
3. **Valid-pixel-count + AOI-area reporting**: AOI area in m² plus the count of valid (unmasked)
   pixels over the AOI.

**Legacy reference** (`legacy:ravi_dialog.py`, `legacy:modules/coordinate_capture.py`):

- `point_calculate_timeseries` / `plot_timeseries_points` — multi-point series + Plotly line plot.
- `feature_calculate_timeseries` / `split_features` / `load_fields` / `feature_info` /
  `plot_timeseries_features` — per-feature series keyed by `attributes_id`.
- `SCL_filter` (the `total_pixels` / `valid_pixels` reducers) and `AOI_coverage_filter`
  (`geometry.area()`) — the area + valid-pixel reporting reproduced here as `calculate_area`.
- `CoordinateCaptureTool` — the concept of capturing WGS84 points off the canvas; here the points
  are supplied directly as `ee.Geometry.Point` instances.

The exact Earth Engine calls (collection id, NDVI as `normalizedDifference(["B8","B4"])`, the
`reduceRegion(mean, scale=10, bestEffort=True).get("index")` per-point/per-feature reducer, the
`reduceRegion(count, scale=10).get("B1")` pixel-count reducer, and `geometry.area()`) are taken
verbatim from the legacy code.

In [ ]:
# Cell 2 — setup
import os
import ee
import pandas as pd
import plotly.express as px

# Service-account auth via the GEE env var (same as dev/testing.ipynb).
credentials = ee.ServiceAccountCredentials(None, os.environ["GEE"])
ee.Initialize(credentials)

# Index name kept as a variable to mirror the dialog's `series_indice` selection.
VEGETATION_INDEX = "NDVI"
print("Earth Engine initialised.")

In [ ]:
# Cell 3 — sample inputs derived from the project AOI (same area as dev/testing.ipynb).
import zipfile

import geopandas as gpd


def load_aoi_from_shapefile(shapefile_path):
    """Load an AOI from a (zipped) shapefile as an ee.FeatureCollection.

    Same loader as dev/testing.ipynb: reproject to EPSG:4326, dissolve to a
    single geometry, strip any Z coordinate, wrap in a FeatureCollection.
    """
    if shapefile_path.endswith(".zip"):
        with zipfile.ZipFile(shapefile_path, "r") as zip_ref:
            shapefile_within_zip = None
            for file in zip_ref.namelist():
                if file.lower().endswith(".shp"):
                    shapefile_within_zip = file
                    break
            if not shapefile_within_zip:
                raise FileNotFoundError(f"No .shp file found inside the zip archive: {shapefile_path}")
            gpd_aoi = gpd.read_file(f"zip://{shapefile_path}/{shapefile_within_zip}")
    else:
        gpd_aoi = gpd.read_file(shapefile_path)

    gpd_aoi = gpd_aoi.to_crs(epsg=4326)

    if gpd_aoi.empty:
        raise ValueError(f"The shapefile at {shapefile_path} does not contain any geometries.")

    if len(gpd_aoi) > 1:
        gpd_aoi = gpd_aoi.dissolve()

    geometry = gpd_aoi.geometry.iloc[0]
    geojson = geometry.__geo_interface__

    if geojson["type"] == "Polygon":
        geojson["coordinates"] = [
            list(map(lambda coord: coord[:2], ring)) for ring in geojson["coordinates"]
        ]
    elif geojson["type"] == "MultiPolygon":
        geojson["coordinates"] = [
            [list(map(lambda coord: coord[:2], ring)) for ring in polygon]
            for polygon in geojson["coordinates"]
        ]

    ee_geometry = ee.Geometry(geojson)
    feature = ee.Feature(ee_geometry)
    return ee.FeatureCollection([feature])


# Date range (legacy `inicio` / `final`).
START_DATE = "2023-01-01"
END_DATE = "2023-06-30"

# AOI from the project shapefile.
aoi_fc = load_aoi_from_shapefile("contorno_area_total.zip")
aoi_geom = aoi_fc.geometry()

# Per-feature workflow: treat the whole AOI as one feature carrying an 'id' attribute
# (legacy `attributes_id`). Swap in a multi-feature vector layer for several fields.
ID_COLUMN = "id"  # legacy `self.attributes_id.currentText()`
FEATURES = ee.FeatureCollection([ee.Feature(aoi_geom, {ID_COLUMN: "area_total"})])

# Per-point workflow: capture point(s) inside the AOI. In the plugin these come from
# CoordinateCaptureTool; here we use the AOI centroid so the point falls within the area.
centroid_coords = aoi_geom.centroid(maxError=1).coordinates().getInfo()  # [lon, lat]
SAMPLE_POINTS = [ee.Geometry.Point(centroid_coords)]

# Sentinel-2 collection, built exactly as in the legacy `sentinel2_update`:
# COPERNICUS/S2_SR_HARMONIZED, date-filtered, with a 'date' string property per image.
sentinel2 = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterDate(START_DATE, END_DATE)
    .filterBounds(FEATURES.geometry())
    .map(lambda image: image.set("date", image.date().format("YYYY-MM-dd")))
)
print("Collection size:", sentinel2.size().getInfo())

In [ ]:
# Cell 4 — shared index + reducer helpers (verbatim from the legacy dialog).

def calculate_vegetation_index(image, index_name):
    """Return the vegetation index as a single band renamed 'index'.

    NDVI is `normalizedDifference(["B8", "B4"])` exactly as in legacy
    `calculate_vegetation_index`.
    """
    if index_name == "NDVI":
        return image.normalizedDifference(["B8", "B4"]).rename("index")
    raise ValueError(f"Unsupported vegetation index: {index_name}")


def calculate_index_with_mean(image, index_name, aoi):
    """Mirror of legacy `calculate_index_with_mean`: AOI-mean of the index, set as 'mean_index'."""
    index_image = calculate_vegetation_index(image, index_name)
    mean_index = index_image.reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=10, bestEffort=True
    ).get("index")
    return image.set({"mean_index": mean_index})


def calculate_timeseries(aoi, name):
    """Mirror of legacy `point_calculate_timeseries` / `feature_calculate_timeseries`.

    Maps the AOI-mean reducer over the collection, drops nulls, and pulls
    dates + mean values with aggregate_array into a (date, name) DataFrame.
    """
    result = sentinel2.map(
        lambda image: calculate_index_with_mean(image, VEGETATION_INDEX, aoi)
    )
    result = result.filter(ee.Filter.notNull(["mean_index"]))
    dates = result.aggregate_array("date").getInfo()
    mean_indices = result.aggregate_array("mean_index").getInfo()
    print(f"Creating DataFrame for {name}")
    return pd.DataFrame({"date": dates, name: mean_indices})

In [ ]:
# Cell 5 — PER-POINT time series (#15).
# One AOI-mean NDVI series per captured point, merged on date, then reshaped long.

point_frames = []
for idx, point in enumerate(SAMPLE_POINTS):
    coords = point.coordinates().getInfo()  # [lon, lat]
    point_id = f"({coords[1]:.4f}, {coords[0]:.4f})"  # legacy label: (lat, long)
    df_point = calculate_timeseries(point, point_id)
    df_point["date"] = pd.to_datetime(df_point["date"])
    point_frames.append(df_point)

df_points_wide = point_frames[0]
for df in point_frames[1:]:
    df_points_wide = pd.merge(df_points_wide, df, on="date", how="outer")
df_points_wide = df_points_wide.sort_values("date").reset_index(drop=True)

# Long/tidy form: (date, point_id, ndvi) — legacy `df.melt(...)` in plot_timeseries_points.
df_points = df_points_wide.melt(
    id_vars="date", var_name="point_id", value_name="ndvi"
).dropna(subset=["ndvi"]).reset_index(drop=True)

display(df_points.head())

fig_points = px.line(
    df_points,
    x="date",
    y="ndvi",
    color="point_id",
    line_dash="point_id",
    title=f"Time Series - {VEGETATION_INDEX} - Points",
)
fig_points.update_layout(yaxis_title=VEGETATION_INDEX, xaxis_title=None)
fig_points.show()

In [ ]:
# Cell 6 — PER-FEATURE time series keyed by the id attribute (#16).
# Legacy `split_features` / `feature_info`: iterate features, one AOI-mean series per feature.

# Pull the feature ids client-side so we can iterate (legacy split_features loop).
feature_ids = FEATURES.aggregate_array(ID_COLUMN).getInfo()
print(f"Total features to process: {len(feature_ids)}")

feature_frames = []
for feature_id in feature_ids:
    # Select this single feature by its id attribute; its geometry is the AOI.
    aoi_feature = FEATURES.filter(ee.Filter.eq(ID_COLUMN, feature_id)).geometry()
    df_feature = calculate_timeseries(aoi_feature, str(feature_id))
    df_feature["date"] = pd.to_datetime(df_feature["date"])
    feature_frames.append(df_feature)

# Merge horizontally on date (legacy feature_info merge), then reshape long.
df_features_wide = feature_frames[0]
for df in feature_frames[1:]:
    df_features_wide = pd.merge(df_features_wide, df, on="date", how="outer")
df_features_wide = df_features_wide.sort_values("date").reset_index(drop=True)

df_features = df_features_wide.melt(
    id_vars="date", var_name="feature_id", value_name="ndvi"
).dropna(subset=["ndvi"]).reset_index(drop=True)

display(df_features.head())

fig_features = px.line(
    df_features,
    x="date",
    y="ndvi",
    color="feature_id",
    line_dash="feature_id",
    title=f"Time Series - {VEGETATION_INDEX} - Features (by '{ID_COLUMN}')",
)
fig_features.update_layout(yaxis_title=VEGETATION_INDEX, xaxis_title=None)
fig_features.show()

In [ ]:
# Cell 7 — AOI area + valid (unmasked) pixel count (#17).
# Mirrors legacy SCL_filter (total_pixels / valid_pixels via reduceRegion count on band 0,
# i.e. 'B1') and AOI_coverage_filter (geometry.area()).

def calculate_area(aoi, image):
    """Report AOI area (m²) and the valid vs total pixel count over the AOI.

    - Area: `aoi.area()` (m²), as in legacy AOI_coverage_filter.
    - total_pixels: count over band 0 ('B1') of the raw image.
    - valid_pixels:  count over band 0 ('B1') of the SCL-masked image.
    Both use reduceRegion(ee.Reducer.count(), scale=10), exactly as legacy SCL_filter.
    """
    # SCL mask: drop the standard cloud / shadow / cirrus classes (3,8,9,10),
    # leaving valid surface pixels (matching the legacy SCL masking concept).
    scl = image.select("SCL")
    mask = ee.Image.constant(1)
    for class_value in (3, 8, 9, 10):
        mask = mask.And(scl.neq(class_value))
    masked_image = image.updateMask(mask)

    total_pixels = (
        image.select(0)
        .reduceRegion(reducer=ee.Reducer.count(), geometry=aoi, scale=10)
        .get("B1")
    )
    valid_pixels = (
        masked_image.select(0)
        .reduceRegion(reducer=ee.Reducer.count(), geometry=aoi, scale=10)
        .get("B1")
    )

    area_m2 = aoi.area().getInfo()
    total = ee.Number(total_pixels).getInfo()
    valid = ee.Number(valid_pixels).getInfo()
    return area_m2, valid, total


# Use the first feature as the reporting AOI and the first image of the collection.
report_aoi = FEATURES.filter(ee.Filter.eq(ID_COLUMN, feature_ids[0])).geometry()
report_image = sentinel2.first()

area_m2, valid, total = calculate_area(report_aoi, report_image)
pct = (valid / total * 100) if total else 0.0

print(f"AOI ('{feature_ids[0]}') area: {area_m2:,.2f} m\u00b2  ({area_m2 / 10_000:,.2f} ha)")
print(f"Valid (unmasked) pixels over AOI: {valid}")
print(f"Total pixels over AOI:            {total}")
print(f"Percentage valid pixels:          {pct:.2f}%")